# Phase 3 — Real Fields: Orientation-Aware Selective Spraying

End-to-end missions on 5 real aerial field images using the geometry pipeline
developed in `phase3_field_geometry.ipynb`.

Each field runs with its own recommended `orientation_deg` and `spray_threshold`
from `data/field_metadata.json` — not a single global setting.

**What's new vs `phase3_real_data`:**
- Per-field orientation matched to visible crop row direction
- Spray threshold filtering: drones transit over sub-threshold cells, sprayer off
- Stop/start spray complexity analysis — segments per strip, on/off toggles per mission
- Overlay GIFs on aerial photos for all 5 fields

**Central questions:**
1. Does orientation-aware stripping visibly follow real crop rows?
2. How much stop/start complexity does selective spraying add?
3. When does MILP earn over greedy on real field geometry?
4. How does the system degrade under failure on a real field?

In [ ]:
import sys, os, json
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

from src.field.ingest import load_image_grid, load_image_as_array
from src.field.generator import generate_strips
from src.optimizer.milp import DroneSpec, assign_strips
from src.optimizer.planner import plan, PlannerMode
from src.simulation.engine import simulate
from src.simulation.metrics import (
    compute_metrics, plot_coverage_over_time,
    monte_carlo_analysis, plot_monte_carlo,
)
from src.viz.renderer import animate

os.makedirs('../results', exist_ok=True)
%matplotlib inline

# --- Load per-field configs from metadata ---
with open('../data/field_metadata.json') as f:
    FIELD_META = {fd['name']: fd for fd in json.load(f)['fields']}

DATA_DIR   = '../data'
N_DRONES   = 3
DOCK_POS   = [(0, 0)]
DRAIN      = 0.3    # % battery per cell -- kept low for 64x64 grids;
                    # higher rates cause transit-loop failures (dock too far from strip)
RECHARGE   = 15     # steps to recharge at dock

FIELD_NAMES = list(FIELD_META.keys())
print(f'{len(FIELD_NAMES)} fields loaded')
for name, fd in FIELD_META.items():
    r = fd['recommended']
    print(f"  {name:<20} size={r['target_size']:>3}  "
          f"orient={r['orientation_deg']:>4}°  "
          f"thresh={r['spray_threshold']}  channel={r['channel']}")

## 1. Field overview — RGB + priority grid

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

all_grids = {}   # name → (grid, meta)

for i, name in enumerate(FIELD_NAMES):
    fd = FIELD_META[name]
    r  = fd['recommended']
    path = f"{DATA_DIR}/{fd['file']}"

    grid, meta = load_image_grid(path, target_size=r['target_size'],
                                 channel=r['channel'], invert=r['invert'])
    bg = load_image_as_array(path, target_size=r['target_size'])
    all_grids[name] = (grid, meta)

    axes[0, i].imshow(bg)
    axes[0, i].set_title(name.replace('field_', ''), fontsize=9)
    axes[0, i].axis('off')

    arr = np.array(grid)
    im  = axes[1, i].imshow(arr, cmap='YlGn', vmin=0, vmax=1, origin='upper')
    axes[1, i].set_title(
        f"{r['channel']}  mean={arr.mean():.2f}  var={arr.var():.4f}",
        fontsize=7
    )
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('RGB', fontsize=9)
axes[1, 0].set_ylabel('Priority grid', fontsize=9)
plt.suptitle('Real field images → priority grids', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Strip geometry — orientation matched to real crop rows

Each field uses its recommended `orientation_deg`. Compare against 0° to see how well
the strips align with visible crop structure in the aerial photo.

- **Coloured lines** = strip traversal paths
- **Filled squares** = spray-active cells
- **× marks** = transit-only (below threshold)
- **Gaps between × clusters** = where the sprayer toggles off mid-strip

In [ ]:
def plot_strip_geometry(strips, field_grid, title='', ax=None):
    arr = np.array(field_grid)
    nrows, ncols = arr.shape
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(arr, cmap='YlGn', vmin=0, vmax=1, alpha=0.45, origin='upper')
    palette = cm.tab20(np.linspace(0, 1, max(len(strips), 1)))
    for i, s in enumerate(strips):
        if not s.cells:
            continue
        ax.plot([c[1] for c in s.cells], [c[0] for c in s.cells],
                '-', color=palette[i % len(palette)], alpha=0.55, linewidth=1.2, zorder=2)
        spray_set = set(map(tuple, s.spray_cells))
        for r, c in s.cells:
            if (r, c) in spray_set:
                ax.plot(c, r, 's', color=palette[i % len(palette)],
                        markersize=5, alpha=0.85, zorder=3)
            else:
                ax.plot(c, r, 'x', color='#aaa', markersize=3.5,
                        markeredgewidth=0.9, alpha=0.5, zorder=3)
    ax.set_title(title, fontsize=8)
    ax.set_xlim(-0.5, ncols - 0.5)
    ax.set_ylim(nrows - 0.5, -0.5)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    return ax


fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for i, name in enumerate(FIELD_NAMES):
    fd  = FIELD_META[name]
    r   = fd['recommended']
    grid, _ = all_grids[name]

    s_rec = generate_strips(grid,
                            orientation_deg=r['orientation_deg'],
                            spray_threshold=r['spray_threshold'])
    s_ref = generate_strips(grid,
                            orientation_deg=0,
                            spray_threshold=r['spray_threshold'])

    spray_rec = sum(len(s.spray_cells) for s in s_rec)
    seg_rec   = sum(len(s.spray_segments) for s in s_rec)
    spray_ref = sum(len(s.spray_cells) for s in s_ref)
    seg_ref   = sum(len(s.spray_segments) for s in s_ref)

    plot_strip_geometry(s_ref, grid,
        title=f'0°  {len(s_ref)} strips  {spray_ref} spray  {seg_ref} segs',
        ax=axes[0, i])
    plot_strip_geometry(s_rec, grid,
        title=f'{r["orientation_deg"]}°  {len(s_rec)} strips  {spray_rec} spray  {seg_rec} segs',
        ax=axes[1, i])

    axes[0, i].set_ylabel(name.replace('field_', ''), fontsize=8)

axes[0, 0].annotate('0° (baseline)', xy=(-0.15, 0.5), xycoords='axes fraction',
                    fontsize=9, rotation=90, va='center')
axes[1, 0].annotate('Recommended', xy=(-0.15, 0.5), xycoords='axes fraction',
                    fontsize=9, rotation=90, va='center')

plt.suptitle('Strip geometry: 0° baseline vs recommended orientation per field', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Mission sweep — all 5 fields

Each field runs with its recommended settings. The sweep computes:
- `spray_coverage_pct` — completed spray cells / total spray cells (the correct metric when a threshold is set)
- `segments_total` — total sprayer on/off events across all strips
- `skip_pct` — % of traversed cells where sprayer was off

In [ ]:
mission_results = {}   # name → dict
all_strips      = {}   # name → strips list (reused later)
all_histories   = {}   # name → state_history

for name in FIELD_NAMES:
    fd   = FIELD_META[name]
    r    = fd['recommended']
    sz   = r['target_size']
    grid, _ = all_grids[name]

    drones = [DroneSpec(id=i) for i in range(N_DRONES)]
    strips = generate_strips(grid,
                             orientation_deg=r['orientation_deg'],
                             spray_threshold=r['spray_threshold'])
    result = plan(strips, drones, mode=PlannerMode.FULL)
    hist   = simulate(
        strips=strips, drones=drones, result=result,
        nrows=sz, ncols=sz,
        battery_drain_per_cell=DRAIN,
        recharge_time_steps=RECHARGE,
        dock_positions=DOCK_POS,
    )
    m = compute_metrics(hist, strips, sz, sz)

    # Spray-specific coverage
    spray_set = set()
    for s in strips:
        spray_set.update(map(tuple, s.spray_cells))
    final_grid = hist[-1]['grid']
    done = sum(1 for rc in spray_set if final_grid[rc[0]][rc[1]] == 2)
    spray_cov = 100 * done / len(spray_set) if spray_set else 0

    total_cells  = sum(len(s.cells) for s in strips)
    total_spray  = sum(len(s.spray_cells) for s in strips)
    total_segs   = sum(len(s.spray_segments) for s in strips)
    skip_pct     = 100 * (1 - total_spray / total_cells) if total_cells else 0

    mission_results[name] = dict(
        n_strips=len(strips), total_cells=total_cells,
        spray_cells=total_spray, segments=total_segs,
        skip_pct=round(skip_pct, 1),
        spray_coverage=round(spray_cov, 1),
        makespan=m['makespan'],
        replan_count=m['replan_count'],
        milp_solve_s=round(result.solve_time, 2),
        milp_status=result.status,
        cells_per_step=m['cells_per_step'],
        battery_series=m['battery_series'],
    )
    all_strips[name]    = strips
    all_histories[name] = hist
    print(f'{name:<22} strips={len(strips):>3}  spray={total_spray:>4}  '
          f'segs={total_segs:>3}  skip={skip_pct:>4.1f}%  '
          f'cov={spray_cov:>5.1f}%  steps={m["makespan"]:>4}  '
          f'replans={m["replan_count"]}  solver={result.solve_time:.2f}s')

In [ ]:
import pandas as pd

rows = []
for name, mr in mission_results.items():
    fd = FIELD_META[name]
    r  = fd['recommended']
    rows.append(dict(
        field=name.replace('field_', ''),
        size=r['target_size'],
        orient=r['orientation_deg'],
        threshold=r['spray_threshold'],
        strips=mr['n_strips'],
        spray_cells=mr['spray_cells'],
        segments=mr['segments'],
        segs_per_strip=round(mr['segments'] / mr['n_strips'], 2),
        skip_pct=mr['skip_pct'],
        spray_cov=mr['spray_coverage'],
        makespan=mr['makespan'],
        replans=mr['replan_count'],
        solve_s=mr['milp_solve_s'],
    ))

df = pd.DataFrame(rows).set_index('field')
display(df)

## 4. GIF animations — all 5 fields (overlay mode)

Aerial photograph as background. Coverage tints overlaid:
- Untouched → transparent (photo shows through)
- Yellow → in-progress
- Gray → complete
- Red → failed/skipped

Notice cells where the drone passes through without the cell changing state — those are the transit-only (below-threshold) cells.

In [ ]:
from matplotlib import rc
rc('animation', html='jshtml')

# Display the most visually interesting field inline
INLINE_FIELD = 'field_multizone'

fd   = FIELD_META[INLINE_FIELD]
sz   = fd['recommended']['target_size']
path = f"{DATA_DIR}/{fd['file']}"
bg   = load_image_as_array(path, target_size=sz)
hist = all_histories[INLINE_FIELD]
skip = max(1, len(hist) // 80)

anim = animate(
    hist[::skip], sz, sz,
    interval_ms=150,
    background_image=bg,
    dock_positions=DOCK_POS,
    show=False,
)
print(f'{INLINE_FIELD}  {len(hist)} steps  frame_skip={skip}  '
      f'{len(hist[::skip])} frames')
anim

In [ ]:
# Save all 5 GIFs
for name in FIELD_NAMES:
    fd   = FIELD_META[name]
    sz   = fd['recommended']['target_size']
    path = f"{DATA_DIR}/{fd['file']}"
    bg   = load_image_as_array(path, target_size=sz)
    hist = all_histories[name]
    skip = max(1, len(hist) // 80)
    out  = f'../results/phase3_{name}_overlay.gif'

    animate(
        hist[::skip], sz, sz,
        interval_ms=140,
        background_image=bg,
        dock_positions=DOCK_POS,
        save_path=out,
        show=False,
    )
    mr = mission_results[name]
    print(f'  {out}  ({len(hist[::skip])} frames  {mr["spray_coverage"]}% spray cov)')

## 5. Stop/start spray complexity

With selective spraying, each strip may have multiple spray segments — the drone
toggles the pump on and off mid-pass. `segs_per_strip > 1` means the drone is
making interrupted passes, which is more realistic but creates a harder scheduling
problem: strip time is now `transit_time + sum(spray_segment_times)`, not a flat rate.

Compare segment distributions across fields and ask: which fields are hardest to plan?

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, name in enumerate(FIELD_NAMES):
    strips = all_strips[name]
    seg_counts = [len(s.spray_segments) for s in strips]

    ax = axes[i]
    max_seg = max(seg_counts) if seg_counts else 1
    bins = range(1, max_seg + 2)
    ax.hist(seg_counts, bins=bins, align='left', color='#1565c0', alpha=0.8, rwidth=0.7)
    ax.axvline(np.mean(seg_counts), color='#e65100', linestyle='--',
               linewidth=1.5, label=f'mean={np.mean(seg_counts):.2f}')
    ax.set_title(name.replace('field_', ''), fontsize=9)
    ax.set_xlabel('Segments per strip')
    if i == 0:
        ax.set_ylabel('Strip count')
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticks(range(1, max_seg + 1))

plt.suptitle('Spray segment distribution per field\n'
             '(segments per strip = number of pump on/off cycles per pass)',
             fontsize=11)
plt.tight_layout()
plt.show()

print('Field                segments  segs/strip  strips w/ gaps (>1 seg)')
print('-' * 60)
for name in FIELD_NAMES:
    strips = all_strips[name]
    sc  = [len(s.spray_segments) for s in strips]
    gap = sum(1 for x in sc if x > 1)
    print(f'{name:<22} {sum(sc):>8}  {np.mean(sc):>10.2f}  '
          f'{gap:>5}/{len(strips)} ({100*gap/len(strips):.0f}%)')

In [ ]:
# Time breakdown: transit vs spray across fields
fig, ax = plt.subplots(figsize=(11, 4))

short_names = [n.replace('field_', '') for n in FIELD_NAMES]
transit_times = []
spray_times   = []

for name in FIELD_NAMES:
    fd = FIELD_META[name]
    r  = fd['recommended']
    strips = all_strips[name]
    grid, _ = all_grids[name]
    arr = np.array(grid)
    sc  = 2.0   # seconds_per_cell default

    tot_transit = sum(len(s.cells) * sc for s in strips)
    tot_spray   = sum(len(s.spray_cells) * sc * s.priority for s in strips)
    transit_times.append(tot_transit)
    spray_times.append(tot_spray)

x = np.arange(len(FIELD_NAMES))
ax.bar(x, transit_times, label='Transit (all cells)', color='#1565c0', alpha=0.8)
ax.bar(x, spray_times,   label='Spray overhead (spray cells × priority)',
       color='#2e7d32', alpha=0.8, bottom=transit_times)
ax.set_xticks(x)
ax.set_xticklabels(short_names)
ax.set_ylabel('Estimated mission time (sim seconds)')
ax.set_title('Transit vs spray time breakdown per field')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. MILP vs greedy — across all fields

Both planners operate on the same strip lists (already generated with recommended settings).
The MILP gets 10 seconds. Greedy is instant.

With orientation-aware selective spraying, strip times are heterogeneous
(transit + spray segments × priority) — this is where MILP earns the most over greedy's
simple least-loaded assignment.

In [ ]:
comparison_rows = []
field_sim_data  = {}   # name → (h_milp, h_greedy, m_milp, m_greedy)

for name in FIELD_NAMES:
    fd     = FIELD_META[name]
    sz     = fd['recommended']['target_size']
    strips = all_strips[name]
    drones = [DroneSpec(id=i) for i in range(N_DRONES)]

    r_milp   = plan(strips, drones, mode=PlannerMode.FULL)
    r_greedy = plan(strips, drones, mode=PlannerMode.HEURISTIC)

    sim_kw = dict(strips=strips, drones=drones, nrows=sz, ncols=sz,
                  battery_drain_per_cell=DRAIN,
                  recharge_time_steps=RECHARGE,
                  dock_positions=DOCK_POS)

    h_milp   = simulate(result=r_milp,   **sim_kw)
    h_greedy = simulate(result=r_greedy, **sim_kw)

    m_milp   = compute_metrics(h_milp,   strips, sz, sz)
    m_greedy = compute_metrics(h_greedy, strips, sz, sz)

    def spray_cov(hist):
        spray_set = set()
        for s in strips:
            spray_set.update(map(tuple, s.spray_cells))
        fg = hist[-1]['grid']
        done = sum(1 for rc in spray_set if fg[rc[0]][rc[1]] == 2)
        return 100 * done / len(spray_set) if spray_set else 0

    field_sim_data[name] = (h_milp, h_greedy, m_milp, m_greedy)
    comparison_rows.append(dict(
        field=name.replace('field_', ''),
        milp_makespan=m_milp['makespan'],
        greedy_makespan=m_greedy['makespan'],
        makespan_ratio=round(m_milp['makespan'] / m_greedy['makespan'], 3),
        milp_spray_cov=round(spray_cov(h_milp), 1),
        greedy_spray_cov=round(spray_cov(h_greedy), 1),
        milp_priority=round(m_milp['priority_coverage'], 4),
        greedy_priority=round(m_greedy['priority_coverage'], 4),
        priority_gain=round(m_milp['priority_coverage'] - m_greedy['priority_coverage'], 4),
        milp_solve_s=round(r_milp.solve_time, 2),
    ))

df_cmp = pd.DataFrame(comparison_rows).set_index('field')
display(df_cmp)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
names_short = [n.replace('field_', '') for n in FIELD_NAMES]
x = np.arange(len(FIELD_NAMES))
w = 0.35

axes[0].bar(x - w/2, df_cmp['milp_makespan'],   w, label='MILP',   color='#1565c0', alpha=0.85)
axes[0].bar(x + w/2, df_cmp['greedy_makespan'], w, label='Greedy', color='#e65100', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(names_short, rotation=15, ha='right')
axes[0].set_title('Makespan (steps)'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - w/2, df_cmp['milp_spray_cov'],   w, label='MILP',   color='#1565c0', alpha=0.85)
axes[1].bar(x + w/2, df_cmp['greedy_spray_cov'], w, label='Greedy', color='#e65100', alpha=0.85)
axes[1].set_ylim(0, 105)
axes[1].set_xticks(x); axes[1].set_xticklabels(names_short, rotation=15, ha='right')
axes[1].set_title('Spray coverage (%)'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

colours = ['#2e7d32' if v >= 0 else '#c62828' for v in df_cmp['priority_gain']]
axes[2].bar(names_short, df_cmp['priority_gain'], color=colours, alpha=0.85)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('Priority gain: MILP − Greedy\n(green = MILP wins)')
axes[2].set_xticklabels(names_short, rotation=15, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('MILP vs Greedy — orientation-aware selective spraying', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Budget sensitivity — field_multizone

`field_multizone` has the highest priority variance — three distinct zones with
very different densities after `pseudo_ndvi`. Strip times are heterogeneous because
each zone has a different priority weight.

Sweep solver budget 0.1s → 15s. The crossover point marks the minimum viable budget
below which greedy should always be used instead.

In [ ]:
SWEEP_FIELD  = 'field_multizone'
fd_sw        = FIELD_META[SWEEP_FIELD]
sz_sw        = fd_sw['recommended']['target_size']
strips_sw    = all_strips[SWEEP_FIELD]
drones_sw    = [DroneSpec(id=i) for i in range(N_DRONES)]

r_greedy_sw  = plan(strips_sw, drones_sw, mode=PlannerMode.HEURISTIC)
h_greedy_sw  = simulate(result=r_greedy_sw, strips=strips_sw, drones=drones_sw,
                        nrows=sz_sw, ncols=sz_sw,
                        battery_drain_per_cell=DRAIN,
                        recharge_time_steps=RECHARGE,
                        dock_positions=DOCK_POS)
m_greedy_sw  = compute_metrics(h_greedy_sw, strips_sw, sz_sw, sz_sw)
greedy_ms    = m_greedy_sw['makespan']

budgets      = [0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0, 15.0]
sweep_rows   = []

for budget in budgets:
    r = assign_strips(strips_sw, drones_sw,
                      objective_mode='makespan',
                      time_limit_seconds=budget)
    if not any(r.assignment.values()):
        continue
    h = simulate(result=r, strips=strips_sw, drones=drones_sw,
                 nrows=sz_sw, ncols=sz_sw,
                 battery_drain_per_cell=DRAIN,
                 recharge_time_steps=RECHARGE,
                 dock_positions=DOCK_POS)
    m = compute_metrics(h, strips_sw, sz_sw, sz_sw)
    sweep_rows.append(dict(
        budget_s=budget,
        status=r.status,
        makespan=m['makespan'],
        ratio=round(m['makespan'] / greedy_ms, 3),
        priority=round(m['priority_coverage'], 4),
    ))
    print(f'  budget={budget:>5.2f}s  status={r.status:<10}  '
          f'makespan={m["makespan"]:>4}  vs greedy {greedy_ms}  '
          f'ratio={m["makespan"]/greedy_ms:.3f}')

print(f'\nGreedy baseline: {greedy_ms} steps')

In [ ]:
budgets_p = [r['budget_s'] for r in sweep_rows]
makespans = [r['makespan'] for r in sweep_rows]
ratios    = [r['ratio']    for r in sweep_rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(budgets_p, makespans, 'o-', color='#1565c0', linewidth=2, markersize=8,
         label='MILP makespan')
ax1.axhline(greedy_ms, color='#e65100', linewidth=2, linestyle='--',
            label=f'Greedy ({greedy_ms} steps)')
ax1.fill_between(budgets_p, makespans, greedy_ms,
                 where=[m > greedy_ms for m in makespans],
                 alpha=0.15, color='#e65100', label='Greedy wins zone')
ax1.fill_between(budgets_p, makespans, greedy_ms,
                 where=[m <= greedy_ms for m in makespans],
                 alpha=0.15, color='#1565c0', label='MILP wins zone')
ax1.set_xlabel('Solver budget (s)')
ax1.set_ylabel('Makespan (steps)')
ax1.set_title(f'Makespan vs solver budget — {SWEEP_FIELD}')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2.plot(budgets_p, ratios, 's-', color='#6a1b9a', linewidth=2, markersize=8)
ax2.axhline(1.0, color='black', linewidth=1, linestyle='--', label='Parity (ratio=1.0)')
ax2.set_xlabel('Solver budget (s)')
ax2.set_ylabel('MILP makespan / Greedy makespan')
ax2.set_title('Makespan ratio vs budget\n(< 1.0 = MILP wins)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.suptitle(f'Budget sensitivity — {SWEEP_FIELD}  '
             f'(orient={fd_sw["recommended"]["orientation_deg"]}°  '
             f'thresh={fd_sw["recommended"]["spray_threshold"]})',
             fontsize=11)
plt.tight_layout()
plt.show()

## 8. Failure injection — field_multizone

Two scripted failures mid-mission: battery (drone 0, early) and mechanical (drone 2, mid).
Compare MILP vs greedy recovery: which assignment survives failure better?

In [ ]:
FOCUS = 'field_multizone'
fd_f  = FIELD_META[FOCUS]
sz_f  = fd_f['recommended']['target_size']
strips_f = all_strips[FOCUS]
drones_f = [DroneSpec(id=i) for i in range(N_DRONES)]

r_milp_f   = plan(strips_f, drones_f, mode=PlannerMode.FULL)
r_greedy_f = plan(strips_f, drones_f, mode=PlannerMode.HEURISTIC)

m_clean = mission_results[FOCUS]['makespan']
fail_t1 = m_clean // 4
fail_t2 = m_clean // 2

FAILURE_EVENTS = [
    {'timestep': fail_t1, 'drone_id': 0, 'type': 'battery'},
    {'timestep': fail_t2, 'drone_id': 2, 'type': 'mechanical'},
]
print(f'Failures: battery t={fail_t1}, mechanical t={fail_t2}  (clean makespan={m_clean})')

sim_kw = dict(
    strips=strips_f, drones=drones_f, nrows=sz_f, ncols=sz_f,
    failure_events=FAILURE_EVENTS,
    battery_drain_per_cell=DRAIN,
    recharge_time_steps=RECHARGE,
    dock_positions=DOCK_POS,
)
h_fail_milp   = simulate(result=r_milp_f,   **sim_kw)
h_fail_greedy = simulate(result=r_greedy_f, **sim_kw)

m_fail_milp   = compute_metrics(h_fail_milp,   strips_f, sz_f, sz_f)
m_fail_greedy = compute_metrics(h_fail_greedy, strips_f, sz_f, sz_f)

def spray_cov_from(hist, strips):
    sp = set()
    for s in strips:
        sp.update(map(tuple, s.spray_cells))
    fg = hist[-1]['grid']
    return 100 * sum(1 for rc in sp if fg[rc[0]][rc[1]] == 2) / len(sp)

print(f'\n{"":25} {"MILP":>12} {"Greedy":>12}')
print(f'{"Makespan":25} {m_fail_milp["makespan"]:>12} {m_fail_greedy["makespan"]:>12}')
print(f'{"Spray coverage":25} {spray_cov_from(h_fail_milp, strips_f):>11.1f}% {spray_cov_from(h_fail_greedy, strips_f):>11.1f}%')
print(f'{"Replan events":25} {m_fail_milp["replan_count"]:>12} {m_fail_greedy["replan_count"]:>12}')
print(f'{"Time to recovery":25} {str(m_fail_milp["time_to_recovery"]):>12} {str(m_fail_greedy["time_to_recovery"]):>12}')

print('\nEvent log (MILP):')
for s in h_fail_milp:
    if s['event']:
        print(f'  t={s["timestep"]:4d}: {s["event"]}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
spray_set_f = set()
for s in strips_f:
    spray_set_f.update(map(tuple, s.spray_cells))
total_spray_f = len(spray_set_f)

plot_coverage_over_time(
    runs={
        f'MILP + failures  (cov={spray_cov_from(h_fail_milp, strips_f):.1f}%)':   m_fail_milp['cells_per_step'],
        f'Greedy + failures (cov={spray_cov_from(h_fail_greedy, strips_f):.1f}%)': m_fail_greedy['cells_per_step'],
        f'MILP clean        (cov={spray_cov_from(all_histories[FOCUS], strips_f):.1f}%)': mission_results[FOCUS]['cells_per_step'],
    },
    total_cells=sz_f * sz_f,
    title=f'Coverage over time — {FOCUS}  (failures at t={fail_t1}, t={fail_t2})',
    ax=ax,
)
for t in [fail_t1, fail_t2]:
    ax.axvline(t, color='red', linestyle=':', alpha=0.7, linewidth=1.5)
ax.text(fail_t1, ax.get_ylim()[1] * 0.9, 'battery', color='red', fontsize=7, ha='center')
ax.text(fail_t2, ax.get_ylim()[1] * 0.9, 'mechanical', color='red', fontsize=7, ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# Failure overlay GIF
path_f = f"{DATA_DIR}/{fd_f['file']}"
bg_f   = load_image_as_array(path_f, target_size=sz_f)
skip_f = max(1, len(h_fail_milp) // 80)

anim_fail = animate(
    h_fail_milp[::skip_f], sz_f, sz_f,
    interval_ms=150,
    background_image=bg_f,
    dock_positions=DOCK_POS,
    save_path=f'../results/phase3_{FOCUS}_failure.gif',
    show=False,
)
print(f'Saved: results/phase3_{FOCUS}_failure.gif  ({len(h_fail_milp[::skip_f])} frames)')
anim_fail

## 9. Monte Carlo worst-case — field_multizone

Random failure injection across 30 runs. With orientation-aware selective spraying
the strip time distribution is wider (heterogeneous transit + spray costs),
so the MILP assignment is more sensitive to losing a drone mid-mission.

In [ ]:
mc = monte_carlo_analysis(
    strips=strips_f, drones=drones_f, result=r_milp_f,
    nrows=sz_f, ncols=sz_f,
    n_runs=30,
    failure_prob_per_drone=0.3,
    battery_drain_range=(0.0, 3.0),
    replan_objective='makespan',
    seed=42,
)

print(f'Monte Carlo — {FOCUS}  (30 runs, p_fail=0.30, drain 0–3%/cell)')
print(f"  Coverage  mean={mc['coverage_pct_mean']}%  "
      f"p5={mc['coverage_pct_p5']}%  p95={mc['coverage_pct_p95']}%")
print(f"  Recovery  mean={mc['time_to_recovery_mean']} steps  "
      f"p95={mc['time_to_recovery_p95']} steps")
print(f"  Worst-case coverage floor: {mc['coverage_pct_p5']}%")

fig, ax = plt.subplots(figsize=(8, 4))
plot_monte_carlo(mc,
    title=f'Monte Carlo — {FOCUS}  (30 runs)  '
          f'orient={fd_f["recommended"]["orientation_deg"]}°  '
          f'thresh={fd_f["recommended"]["spray_threshold"]}',
    ax=ax)
plt.tight_layout()
plt.savefig(f'../results/phase3_{FOCUS}_montecarlo.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: results/phase3_{FOCUS}_montecarlo.png')

## Summary

| Field | Orientation | Skip% | Segs/strip | Spray cov | MILP vs Greedy |
|---|---|---|---|---|---|
| multizone | 135° | high (bare soil zones) | ~1 (zone boundaries clean) | varies | MILP wins on heterogeneous strip times |
| orchard | 35° | moderate (path gaps) | >1 (tree canopy gaps between rows) | high | mixed |
| elevation | 0° | low (most zones active) | ~1 | moderate | even |
| meadow | 0° | moderate (tree border) | ~1 | high | even |
| paddies | 90° | high (flooded centres) | >1 (green borders only) | high | MILP wins |

**Key observations:**
- Strip orientation aligned to real crop rows produces visually cleaner GIFs — the drone traversal pattern matches the field structure
- Stop/start spraying is most pronounced on `paddies` and `orchard` where crop is spatially discontinuous within a pass
- MILP earns the most over greedy on fields with heterogeneous strip times — `multizone` (three priority zones) and `paddies` (high-threshold, border-only spraying)
- Worst-case coverage from Monte Carlo bounds the risk for operators: the system is robust to single drone failure but degrades with two simultaneous failures